# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** 24_ceo_optimizer    
**Author:** Jasper Cluistra   
**Last Updated:** 2026-02-27
### Cost and ILP optimization workflow
**Goal:** Generate a cost matrix for the geometry using the timber datasets, then use ILP to find the best matches.
**Inputs:**
*   CSV timber dataset
*   Digital geometry

**Outputs:**
*   Best match for each structural element

# IMPORT

In [ ]:
import config
import pandas as pd
import numpy as np
import json
import random
import pulp
import importlib

# Import custom modules
import c11_params
from geometry import generate_sample_vertices
from c23_reconstruction import reconstruct_edges
from c25_structural_check import compute_utilization_outputs
from c26_cost_calculation import prepare_stock_cost_inputs, build_cost_matrix
from c24_fitness_aggregation import (
    evaluate_milp_solution,
    print_fitness_breakdown,
    get_normalization_constants
)
from normalization_constants import get_normalization_constants as get_norm_constants

print("✓ All imports loaded successfully")

System loaded successfully.

Code is running locally from: thesis_generative_timber
Data connected to OneDrive: 2.2 - 2.4

parameters loaded from c:\Users\Jasper\Documents\PyRepo\thesis_generative_timber\c11_params.py
GRID: 2x2, EDGE_LENGTH: 3.0, LAYER_HEIGHT: 1.5, DIVISIONS: 8, NUM_SAMPLES: 20000

✓ All imports loaded successfully


# MULTI-OBJECTIVE FITNESS FUNCTION

## Overview

The fitness function balances three competing objectives:

$$F(\mathbf{x}) = \omega_1 \left( \frac{f_{inner}^*(\mathbf{x})}{\mathcal{C}_{max}} \right) - \omega_2 \left( \frac{\mathcal{R}(\mathbf{x}, \mathbf{y}^*)}{\mathcal{R}_{max}} \right) + \omega_3 \left( \frac{\mathcal{W}(\mathbf{x}, \mathbf{y}^*)}{\mathcal{W}_{max}} \right)$$

- **$f_{inner}^*$**: MILP cost (kg CO2e) — penalizes virgin material and waste
- **$\mathcal{R}$**: Reuse rate (%) — reward for using reclaimed timber (subtracted, so higher reuse = better)
- **$\mathcal{W}$**: Total waste (m³) — penalizes inefficient cutting
- **$\omega_i$**: Weight coefficients to tune priorities

All metrics are normalized to [0, 1] using precomputed dataset-driven extremes ($\mathcal{C}_{max}$, $\mathcal{R}_{max}$, $\mathcal{W}_{max}$).

## Configuration

**Design philosophy**: The MILP cost matrix already accounts for all physical and geometric infeasibilities (by returning ∞ for impossible assignments). This means the upper-level fitness function does NOT need a conditional fallback loop. Instead, we can directly minimize a weighted multi-objective sum.

**Sign convention**:
- Positive coefficient on cost: higher MILP cost → higher (worse) fitness
- Negative coefficient on reuse: higher reuse rate → lower (better) fitness ✓
- Positive coefficient on waste: higher waste → higher (worse) fitness

**Typical design trade-offs**:
- High ω₁ (cost weight): prioritize LCA minimization across all materials
- High ω₂ (reuse weight): prioritize reclaimed material recovery
- High ω₃ (waste weight): prioritize cutting efficiency

In [2]:
# ==========================================
# 1. LOAD NORMALIZATION CONSTANTS
# ==========================================
# These are precomputed from dataset extremes and allow all metrics
# to be scaled to [0, 1] range for fair comparison

norm_constants = get_normalization_constants()
C_MAX = norm_constants['C_max']
R_MAX = norm_constants['R_max']
W_MAX = norm_constants['W_max']

print("Normalization Constants Loaded:")
print(f"  C_max (worst-case cost):   {C_MAX} kg CO2e")
print(f"  R_max (100% reuse rate):   {R_MAX} %")
print(f"  W_max (max waste):         {W_MAX} m³")

# ==========================================
# 2. CONFIGURE WEIGHTS FOR MULTI-OBJECTIVE
# ==========================================
# Adjust these weights to balance priorities:
#   omega_1: cost (1.0 = cost is primary driver)
#   omega_2: reuse rate (0.5 = moderate; increase to prioritize timber recovery)
#   omega_3: waste (0.5 = moderate; increase to prioritize cutting efficiency)
#
# STRATEGY 1 (default, cost-dominant):
#   weights = (1.0, 0.5, 0.5)  # Cost is 2x more important than reuse/waste
#
# STRATEGY 2 (reuse-prioritized):
#   weights = (1.0, 2.0, 0.25)  # Maximize timber recovery, accept higher cost
#
# STRATEGY 3 (balanced):
#   weights = (1.0, 1.0, 1.0)  # All objectives equally important

WEIGHT_CONFIG = {
    'omega_1': 1.0,   # Cost weight (higher = prioritize LCA)
    'omega_2': 0.5,   # Reuse weight (higher = prioritize reclaimed timber)
    'omega_3': 0.5,   # Waste weight (higher = prioritize cutting efficiency)
    'strategy': 'cost-dominant'  # Easy reference for logging
}

print(f"\nWeight Configuration ({WEIGHT_CONFIG['strategy']}):")
print(f"  ω₁ (cost):  {WEIGHT_CONFIG['omega_1']}")
print(f"  ω₂ (reuse): {WEIGHT_CONFIG['omega_2']}")
print(f"  ω₃ (waste): {WEIGHT_CONFIG['omega_3']}")
print(f"\nTo change strategy, edit WEIGHT_CONFIG before running optimize_design()")
print("=" * 70)

Normalization Constants Loaded:
  C_max (worst-case cost):   8.0 kg CO2e
  R_max (100% reuse rate):   100.0 %
  W_max (max waste):         0.4 m³

Weight Configuration (cost-dominant):
  ω₁ (cost):  1.0
  ω₂ (reuse): 0.5
  ω₃ (waste): 0.5

To change strategy, edit WEIGHT_CONFIG before running optimize_design()


## Design Evaluation Pipeline

The function below orchestrates the full pipeline: geometry → surrogate model → structural check → cost matrix → MILP → fitness aggregation.

In [ ]:
# ==========================================
# MAIN EVALUATION FUNCTION
# ==========================================

def evaluate_design(genes: dict, 
                   timber_stock_df: pd.DataFrame = None,
                   model_prefix: str = "data_4_0000",
                   verbose: bool = False) -> dict:
    """
    Comprehensive design evaluation pipeline combining all workflow steps.
    
    Pipeline:
    1. Generate 3D geometry from design genes
    2. Extract structural forces using pre-trained surrogate model
    3. Compute structural utilization (Eurocode 5)
    4. Build cost matrix (LCA-based, accounts for infeasibilities)
    5. Solve MILP to find optimal timber assignments
    6. Extract metrics (cost, reuse, waste) and compute fitness
    
    Args:
        genes: Dict with design parameters (e.g. from search space JSON)
        timber_stock_df: Pre-loaded timber inventory (optional; loads if None)
        model_prefix: Surrogate model identifier (default: data_4_0000)
        verbose: If True, prints detailed progress information
    
    Returns:
        dict with:
          - 'fitness': Multi-objective fitness value (to minimize)
          - 'cost': MILP cost in kg CO2e
          - 'reuse_rate': Percentage of reclaimed timber used
          - 'waste': Total waste volume in m³
          - And detailed metric breakdown
    
    Error handling:
        Returns failed design with fitness=∞ if any step fails (no geometry,
        MILP infeasible, etc.). This allows EA to automatically reject bad designs.
    """
    
    try:
        # ==========================================
        # STEP 1: GENERATE GEOMETRY
        # ==========================================
        if verbose:
            print("[1/6] Generating 3D geometry...")
        
        vertices_list = generate_sample_vertices(sample_id=0, params=genes)
        df_vertices = pd.DataFrame(vertices_list)
        df_edges = reconstruct_edges(c11_params.GRID_CELLS_X, c11_params.GRID_CELLS_Y)
        
        if df_vertices.empty or df_edges.empty:
            raise ValueError("Geometry generation failed: empty vertices or edges")
        
        if verbose:
            print(f"   ✓ Geometry: {len(df_vertices)} vertices, {len(df_edges)} edges")
        
        # ==========================================
        # STEP 2: SURROGATE MODEL INFERENCE
        # ==========================================
        if verbose:
            print("[2/6] Running surrogate model (GNN inference)...")
        
        # NOTE: Placeholder; real implementation requires:
        # - Loading trained GNN model
        # - Preparing node features (x, y, z, ...)
        # - Running inference
        # - Storing predictions in df_forces
        # For now, assume df_forces comes from previous pipeline
        
        # This step requires loading the model and running inference
        # (Deferred to actual integration with c26_27 workflow)
        
        # ==========================================
        # STEP 3: STRUCTURAL CHECK & UTILIZATION
        # ==========================================
        if verbose:
            print("[3/6] Computing structural utilization...")
        
        # This step requires df_forces from Step 2
        # (Deferred; assumes forces are available)
        
        # ==========================================
        # STEP 4: COST MATRIX GENERATION
        # ==========================================
        if verbose:
            print("[4/6] Building cost matrix...")
        
        # Load timber stock if not provided
        if timber_stock_df is None:
            file_path = config.TIMBER_STOCK_PATH / 'complete_timber.csv'
            timber_stock_df = pd.read_csv(file_path, sep=";")
        
        # Build cost matrix (requires df_slots from structural check)
        # This is a simplified placeholder
        
        # ==========================================
        # STEP 5: MILP SOLVER
        # ==========================================
        if verbose:
            print("[5/6] Solving MILP assignment problem...")
        
        # MILP solver logic would be embedded here (from c26_27 notebook)
        # Returns: df_results, enriched_stock, prob.objective value
        
        # Placeholder: simulate successful MILP solve
        # In production: run full MILP cell logic here
        
        # ==========================================
        # STEP 6: FITNESS AGGREGATION
        # ==========================================
        if verbose:
            print("[6/6] Aggregating fitness metrics...")
        
        # This step requires actual MILP results
        # In production:
        # fitness_result = evaluate_milp_solution(
        #     milp_results_df=df_results,
        #     enriched_stock_df=enriched_stock,
        #     df_slots=df_slots,
        #     milp_objective_value=pulp.value(prob.objective),
        #     weights=(WEIGHT_CONFIG['omega_1'],
        #              WEIGHT_CONFIG['omega_2'],
        #              WEIGHT_CONFIG['omega_3'])
        # )
        # 
        # if verbose:
        #     print_fitness_breakdown(fitness_result)
        # 
        # return fitness_result
        
        # For now, return a placeholder success structure
        return {
            'fitness': 0.5,  # Placeholder
            'cost_raw': 4.0,
            'reuse_rate': 50.0,
            'waste_total': 0.2,
            'cost_norm': 0.5,
            'reuse_norm': 0.5,
            'waste_norm': 0.5,
            'weights': (WEIGHT_CONFIG['omega_1'],
                       WEIGHT_CONFIG['omega_2'],
                       WEIGHT_CONFIG['omega_3']),
            'status': 'placeholder'
        }
    
    except Exception as e:
        if verbose:
            print(f"\n✗ Evaluation failed: {str(e)}")
        
        # Return a failed design with infinite fitness (EA will reject it)
        return {
            'fitness': np.inf,
            'cost_raw': np.inf,
            'reuse_rate': 0.0,
            'waste_total': np.inf,
            'cost_norm': 1.0,
            'reuse_norm': 0.0,
            'waste_norm': 1.0,
            'weights': (WEIGHT_CONFIG['omega_1'],
                       WEIGHT_CONFIG['omega_2'],
                       WEIGHT_CONFIG['omega_3']),
            'error': str(e)
        }


print("✓ evaluate_design() function ready")

## Future Integration: Evolutionary Algorithm

The `evaluate_design()` function is designed to integrate seamlessly with any evolutionary algorithm (EA) or derivative-free optimizer:

### Example pattern (pseudo-code):
```python
# Load search space and timber stock
with open(config.DATA_IO_PATH / "search_space.json") as f:
    search_space = json.load(f)
timber_stock_df = load_timber_stock()

# Define EA fitness evaluator
def fitness_func(individual):
    genes = {param: individual[i] for i, param in enumerate(search_space.keys())}
    result = evaluate_design(genes, timber_stock_df=timber_stock_df, verbose=False)
    return result['fitness']  # Single value to minimize

# Run EA (e.g., DEAP, scipy.optimize.differential_evolution, etc.)
# best_design, best_fitness = run_evolutionary_algorithm(fitness_func, ...)

# Evaluate best design with verbose output
best_result = evaluate_design(best_design, timber_stock_df=timber_stock_df, verbose=True)
print_fitness_breakdown(best_result)
```

### Key design decisions:
1. **Single fitness value**: EA receives `result['fitness']`, a scalar to minimize
2. **Error handling**: Failed designs return `fitness=∞`, automatically rejected by EA
3. **Reproducibility**: Weight config and normalization constants are logged
4. **Debugging**: Set `verbose=True` to trace pipeline steps

## Testing & Validation

Unit tests to verify the fitness function logic without running the full pipeline.

In [ ]:
from c24_fitness_aggregation import (
    normalize_metrics, 
    fitness_function_multi_objective,
    print_fitness_breakdown
)

# ==========================================
# TEST 1: Normalization Function
# ==========================================
print("=" * 70)
print("TEST 1: Normalization")
print("=" * 70)

# Scenario 1: Mid-range values
cost_1, reuse_1, waste_1 = normalize_metrics(
    cost=4.0,        # 50% of C_max (8.0)
    reuse_rate=50.0, # 50% of R_max (100%)
    waste=0.2        # 50% of W_max (0.4)
)
print(f"\nScenario 1 (mid-range):")
print(f"  Raw: cost=4.0, reuse=50%, waste=0.2")
print(f"  Normalized: cost={cost_1:.3f}, reuse={reuse_1:.3f}, waste={waste_1:.3f}")
assert 0.45 < cost_1 < 0.55, "Cost normalization failed"
assert 0.45 < reuse_1 < 0.55, "Reuse normalization failed"
assert 0.45 < waste_1 < 0.55, "Waste normalization failed"
print("  ✓ PASS")

# Scenario 2: Excellent design (high reuse, low cost/waste)
cost_2, reuse_2, waste_2 = normalize_metrics(
    cost=2.0,        # 25% of C_max
    reuse_rate=100.0,# 100% reuse
    waste=0.05       # 12.5% of W_max
)
print(f"\nScenario 2 (excellent: high reuse):")
print(f"  Normalized: cost={cost_2:.3f}, reuse={reuse_2:.3f}, waste={waste_2:.3f}")
assert 0 <= cost_2 <= 1 and 0 <= reuse_2 <= 1 and 0 <= waste_2 <= 1
print("  ✓ PASS")

# ==========================================
# TEST 2: Fitness Formula
# ==========================================
print("\n" + "=" * 70)
print("TEST 2: Multi-Objective Fitness Formula")
print("=" * 70)

weights_default = (1.0, 0.5, 0.5)

# Scenario: Excellent design (should be NEGATIVE/low)
fitness_excellent = fitness_function_multi_objective(
    cost_norm=0.25,
    reuse_norm=1.0,   # High reuse
    waste_norm=0.1,
    weights=weights_default
)
print(f"\nExcellent design (high reuse, low cost/waste):")
print(f"  F = 1.0×0.25 - 0.5×1.0 + 0.5×0.1")
print(f"  F = 0.25 - 0.5 + 0.05 = {fitness_excellent:.3f}")
print(f"  Expected: <0 (reuse benefit outweighs cost)")
assert fitness_excellent < 0, "Excellent design should have negative fitness"
print("  ✓ PASS (negative fitness = good design)")

# Scenario: Poor design (all virgin, high waste)
fitness_poor = fitness_function_multi_objective(
    cost_norm=0.8,
    reuse_norm=0.0,   # No reuse
    waste_norm=0.9,
    weights=weights_default
)
print(f"\nPoor design (virgin timber, high waste):")
print(f"  F = 1.0×0.8 - 0.5×0.0 + 0.5×0.9")
print(f"  F = 0.8 - 0 + 0.45 = {fitness_poor:.3f}")
print(f"  Expected: >0.5 (penalized for waste and cost)")
assert fitness_poor > 0.5, "Poor design should have high positive fitness"
print("  ✓ PASS (positive fitness = bad design)")

# ==========================================
# TEST 3: Weight Sensitivity
# ==========================================
print("\n" + "=" * 70)
print("TEST 3: Weight Sensitivity Analysis")
print("=" * 70)

test_case = {
    'cost_norm': 0.5,
    'reuse_norm': 0.5,
    'waste_norm': 0.5
}

strategies = [
    ("cost-dominant", (1.0, 0.5, 0.5)),
    ("reuse-prioritized", (1.0, 2.0, 0.25)),
    ("balanced", (1.0, 1.0, 1.0)),
]

print(f"\nScenario: cost=0.5, reuse=0.5, waste=0.5")
for name, weights in strategies:
    f = fitness_function_multi_objective(*test_case.values(), weights=weights)
    print(f"  {name:20s} ω={weights} → F={f:7.3f}")

print("  ✓ PASS (weights properly control trade-offs)")

# ==========================================
# TEST 4: Configuration Check
# ==========================================
print("\n" + "=" * 70)
print("TEST 4: Configuration Status")
print("=" * 70)

print(f"\nNormalization Constants:")
print(f"  C_max: {C_MAX} kg CO2e")
print(f"  R_max: {R_MAX} %")
print(f"  W_max: {W_MAX} m³")

print(f"\nActive Weight Strategy:")
print(f"  Strategy: {WEIGHT_CONFIG['strategy']}")
print(f"  ω₁: {WEIGHT_CONFIG['omega_1']} (cost)")
print(f"  ω₂: {WEIGHT_CONFIG['omega_2']} (reuse)")
print(f"  ω₃: {WEIGHT_CONFIG['omega_3']} (waste)")

print("\n✓ All tests passed! Fitness function ready for integration.")
print("=" * 70)

TEST 1: Normalization

Scenario 1 (mid-range):
  Raw: cost=4.0, reuse=50%, waste=0.2
  Normalized: cost=0.500, reuse=0.500, waste=0.500
  ✓ PASS

Scenario 2 (excellent: high reuse):
  Normalized: cost=0.250, reuse=1.000, waste=0.125
  ✓ PASS

TEST 2: Multi-Objective Fitness Formula

Excellent design (high reuse, low cost/waste):
  F = 1.0×0.25 - 0.5×1.0 + 0.5×0.1
  F = 0.25 - 0.5 + 0.05 = -0.200
  Expected: <0 (reuse benefit outweighs cost)
  ✓ PASS (negative fitness = good design)

Poor design (virgin timber, high waste):
  F = 1.0×0.8 - 0.5×0.0 + 0.5×0.9
  F = 0.8 - 0 + 0.45 = 1.250
  Expected: >0.5 (penalized for waste and cost)
  ✓ PASS (positive fitness = bad design)

TEST 3: Weight Sensitivity Analysis

Scenario: cost=0.5, reuse=0.5, waste=0.5
  cost-dominant        ω=(1.0, 0.5, 0.5) → F=  0.500
  reuse-prioritized    ω=(1.0, 2.0, 0.25) → F= -0.375
  balanced             ω=(1.0, 1.0, 1.0) → F=  0.500
  ✓ PASS (weights properly control trade-offs)

TEST 4: Configuration Status

No